# YOLOv8s AFPN + ATFL + Inner-IoU + DGLR-Head on Kaggle

这份 notebook 用于在 Kaggle 上短训验证我们最后一刀的模块：

- `YOLOv8s + AFPN`
- `ATFL`
- `Inner-IoU`
- `DGLR-Head (Drill-pipe Glare-aware Localization Refinement Head)`

设计目标：

- 针对 `低高度 drill_pipe`
- 更贴合 `暗场 + 强光斑` 的 hard images
- 只在 `P3/P4` 修 `box + quality`，不动 `cls`
- 先跑 `50 epoch` 做短训筛选，有效果再长训


In [ ]:
!nvidia-smi


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tuanziya666/yolov8s_kuangjing.git"
REPO_DIR = Path("/kaggle/working/yolov8s_kuangjing")

if REPO_DIR.exists():
    %cd /kaggle/working/yolov8s_kuangjing
    !git fetch origin
    !git pull --ff-only origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q -e .


In [ ]:
# 自动查找 Kaggle 上的 3 类数据集，并生成本地 data.yaml
from pathlib import Path
import yaml

INPUT_ROOT = Path('/kaggle/input')
candidate_roots = []

for p in INPUT_ROOT.rglob('*'):
    if p.is_dir() and (p / 'images' / 'train').exists() and (p / 'labels' / 'train').exists():
        candidate_roots.append(p)

if not candidate_roots:
    raise FileNotFoundError('没有在 /kaggle/input 下找到包含 images/train 和 labels/train 的数据集目录。')

def score_root(path: Path):
    score = 0
    name = path.name.lower()
    if '3class' in name or '3cls' in name:
        score += 10
    y = path / 'data.yaml'
    if y.exists():
        try:
            cfg = yaml.safe_load(y.read_text(encoding='utf-8'))
            if int(cfg.get('nc', -1)) == 3:
                score += 20
        except Exception:
            pass
    return score, str(path)

candidate_roots = sorted(set(candidate_roots), key=score_root, reverse=True)
DATASET_ROOT = candidate_roots[0]
print('DATASET_ROOT =', DATASET_ROOT)

DATA_CFG = REPO_DIR / 'ultralytics' / 'cfg' / 'datasets' / 'coalmine3_kaggle_dglr.yaml'
data_cfg = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': ['chuck', 'gripper', 'drill_pipe'],
}
DATA_CFG.write_text(yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(DATA_CFG.read_text(encoding='utf-8'))


In [ ]:
from pathlib import Path

MODEL_CFG = 'ultralytics/cfg/models/v8/yolov8s_afpn.yaml'
PRETRAINED = 'yolov8s.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = '0'
PROJECT = '/kaggle/working/runs/detect'
NAME = 'yolov8s_afpn_atfl_inneriou_dglr_p3p4_3cls_seed42_e50_kaggle'
SEED = 42
PATIENCE = 50

DGLR_LEVELS = 'p3p4'
DGLR_LAMBDA = 0.2
DGLR_SCORE_MODE = 'mul'
DGLR_ALPHA = 0.6
USE_DRILL_QUALITY_WEIGHT = True
DRILL_QUALITY_REFINE = True
TARGET_CLASS_IDS = '2'
DRILL_BASE_WEIGHT = 1.2
DRILL_SMALL_H1 = 0.06
DRILL_SMALL_H2 = 0.09
DRILL_SMALL_W1 = 1.3
DRILL_SMALL_W2 = 1.15

assert Path(MODEL_CFG).exists(), f'Missing model config: {MODEL_CFG}'
assert Path('train_yolov8s_afpn_atfl_inneriou_dglr_head.py').exists(), 'Missing DGLR training script'
print('DATA_CFG =', DATA_CFG)
print('MODEL_CFG =', MODEL_CFG)
print('EPOCHS =', EPOCHS)
print('NAME =', NAME)


In [ ]:
%cd /kaggle/working/yolov8s_kuangjing

import subprocess
import sys

cmd = [
    sys.executable,
    '-u',
    'train_yolov8s_afpn_atfl_inneriou_dglr_head.py',
    '--data', str(DATA_CFG),
    '--model', MODEL_CFG,
    '--pretrained', PRETRAINED,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--device', str(DEVICE),
    '--workers', str(WORKERS),
    '--cache', 'False',
    '--amp', 'True',
    '--seed', str(SEED),
    '--deterministic', 'True',
    '--optimizer', 'SGD',
    '--patience', str(PATIENCE),
    '--project', PROJECT,
    '--name', NAME,
    '--inner-ratio', '0.8',
    '--dglr-levels', DGLR_LEVELS,
    '--dglr-lambda', str(DGLR_LAMBDA),
    '--dglr-score-mode', DGLR_SCORE_MODE,
    '--dglr-alpha', str(DGLR_ALPHA),
    '--use-drill-quality-weight', str(USE_DRILL_QUALITY_WEIGHT),
    '--drill-quality-refine', str(DRILL_QUALITY_REFINE),
    '--dglr-target-class-ids', TARGET_CLASS_IDS,
    '--drill-quality-base-weight', str(DRILL_BASE_WEIGHT),
    '--drill-quality-small-h1', str(DRILL_SMALL_H1),
    '--drill-quality-small-h2', str(DRILL_SMALL_H2),
    '--drill-quality-small-w1', str(DRILL_SMALL_W1),
    '--drill-quality-small-w2', str(DRILL_SMALL_W2),
]

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
import shutil

run_dir = Path(PROJECT) / NAME
full_zip_path = shutil.make_archive(f'/kaggle/working/{NAME}_full', 'zip', root_dir=run_dir)
print('full_zip_path =', full_zip_path)

tmp_dir = Path(f'/kaggle/working/{NAME}_weights_only_tmp')
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir(parents=True, exist_ok=True)

for rel_path in [
    'weights/best.pt',
    'weights/last.pt',
    'results.csv',
    'args.yaml',
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
]:
    src = run_dir / rel_path
    if src.exists():
        dst = tmp_dir / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

weights_zip_path = shutil.make_archive(f'/kaggle/working/{NAME}_weights_only', 'zip', root_dir=tmp_dir)
print('weights_zip_path =', weights_zip_path)
shutil.rmtree(tmp_dir, ignore_errors=True)
